In [3]:
import pandas as pd

# DIA 6

In [6]:
import xmlrpc.client
from datetime import date, timedelta
import pandas as pd

# Mostrar todas las columnas
pd.set_option('display.max_columns', None)

# Mostrar todas las filas (si quieres limitarlo, usa por ejemplo 200)
pd.set_option('display.max_rows', None)

# Ampliar el ancho de cada columna para ver bien el texto
pd.set_option('display.max_colwidth', None)

# Ajustar el ancho de la tabla en consola
pd.set_option('display.width', 0)
# ===============================

import xmlrpc.client
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

# ===============================
# 1. Conexión con Odoo
# ===============================
ODOO_URL = "https://www.donssonusa.com"
db = "donsson-filters-florida-llc"
ODOO_USERNAME = "gbetancourt@donsson.com"
password = "DonssonFloridaCol"   

common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
uid = common.authenticate(db, ODOO_USERNAME, password, {})
if not uid:
    print("Error de autenticación. Verifica las credenciales.")
    exit()

models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")

# ===============================
# 2. Extraer productos
# ===============================

def obtener_campos_modelo(model_name):
    fields = models.execute_kw(
        db, uid, password,
        model_name,
        "fields_get",
        [],
        {"attributes": ["string", "type", "relation", "required"]}
    )

    df = (
        pd.DataFrame.from_dict(fields, orient="index")
        .reset_index()
        .rename(columns={"index": "field_name"})
        .sort_values("field_name")
    )

    return df


In [13]:
df_product_template_fields = obtener_campos_modelo("product.template")

df_product_template_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_product_template.csv")

In [15]:
df_stock_quant_fields = obtener_campos_modelo("stock.quant")

df_stock_quant_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_stock_quant.csv")


In [16]:
df_product_variant_fields = obtener_campos_modelo("product.product")


df_product_variant_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_product_varaint.csv")

In [19]:
# PRODUCTOS
product_fields = [
    "id",
    "name",
    "default_code",
    "image_1920",
    "weight",
    "volume",
    #"brand_id"  # no existe
]

products = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [[]],
    {"fields": product_fields}
)

df_products = pd.DataFrame(products)


## identificar sin foto ni medidas ni codigo

In [26]:
df_sin_foto = df_products[
    df_products["image_1920"].isna() | (df_products["image_1920"] == False)
]

df_sin_codigo = df_products[
    df_products["default_code"].isna() | (df_products["default_code"] == "")
]

df_sin_logistica = df_products[
    (df_products["weight"].isna()) | (df_products["weight"] == 0) |
    (df_products["volume"].isna()) | (df_products["volume"] == 0)
]

def marca_sugerida(nombre):
    nombre = str(nombre).upper()
    if "DA" in nombre:
        return "Donsson"
    elif "BALDWIN" in nombre:
        return "Baldwin"
    else:
        return "Otro"

for df in [df_sin_foto, df_sin_codigo, df_sin_logistica]:
    df["Marca_Sugerida"] = df["name"].apply(marca_sugerida)
    df["Foto_Disponible_Colombia"] = ""





/tmp/ipykernel_26941/1109614719.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Marca_Sugerida"] = df["name"].apply(marca_sugerida)
/tmp/ipykernel_26941/1109614719.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Foto_Disponible_Colombia"] = ""


In [27]:
with pd.ExcelWriter(
    "/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/DIA_6_Diagnostico_Catalogo.xlsx",
    engine="xlsxwriter"
) as writer:
    df_sin_foto.to_excel(writer, sheet_name="Sin_Foto", index=False)
    df_sin_codigo.to_excel(writer, sheet_name="Sin_Internal_Reference", index=False)
    df_sin_logistica.to_excel(writer, sheet_name="Sin_Peso_Medidas", index=False)

In [ ]:
df_carga_codigo = df_sin_codigo[[
    "id",
    "name",
    "default_code"
]].copy()

df_carga_codigo.rename(columns={
    "default_code": "Internal_Reference (A completar)"
}, inplace=True)


In [ ]:
df_carga_marca = df_products[[
    "id",
    "name"
]].copy()

df_carga_marca["Marca (Donsson / Baldwin)"] = df_products.apply(marca_sugerida, axis=1)


In [ ]:
with pd.ExcelWriter(
    "DONSSON MIAMI/salidas/DIA_6_Plantillas_Carga.xlsx",
    engine="xlsxwriter"
) as writer:
    df_carga_codigo.to_excel(writer, sheet_name="Carga_Internal_Reference", index=False)
    df_carga_marca.to_excel(writer, sheet_name="Carga_Marca", index=False)
